In [ ]:
import pandas as pd
import kagglehub
import os
import re
import string

from wordcloud import WordCloud
from langdetect import detect, DetectorFactory
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer

**Data Preprocessing**

In [ ]:
# data import
# original source:https://www.kaggle.com/datasets/nikhileswarkomati/suicide-watch

In [ ]:
# Download latest version
path = kagglehub.dataset_download("nikhileswarkomati/suicide-watch")
print("Path to dataset files:", path)

In [ ]:
all_files = os.listdir(path)
all_files

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
file_path = os.path.join(path, 'Suicide_Detection.csv')
df = pd.read_csv(file_path)
df.sample(5)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# total 232074 rows
df.count()

In [ ]:
df.dtypes

In [ ]:
# check balance of classes
df['class'].value_counts()

In [ ]:
df = df.drop('Unnamed: 0', axis=1)

In [ ]:
df['text_len'] = df['text'].apply(len)

In [ ]:
df['text_len'].max()

In [ ]:
df['text_len'].mean()

In [ ]:
df['text_len'].min()

In [ ]:
df.shape

In [ ]:
# check for "filler" pattern
# "filler" is a valid word, but sometimes used to fill space
filler_data = df[df["text"].str.contains('filler filler', case=False)]
filler_data.sample(10)

**EDA**:explorre the relationship between class type and text length

In [ ]:
plt.figure(figsize=(15, 6))
sns.kdeplot(data=df, x='text_len', hue='class', fill=True, common_norm=False)
plt.title('Density of Text Length (Original Scale)')
plt.xlabel('Length of Text (Characters)')
plt.ylabel('Density')
plt.show()

In [ ]:
# observe: most text length fall within 5000, and suicide type tend to be longer content
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df[df['text_len'] < 5000], x='text_len', hue='class', fill=True)
plt.title('Density of Text Length (under 5000 characters)')
plt.show()

In [ ]:
# Look at the low end
df[df['text_len'] < 20]['text'].sample(20)

In [ ]:
# Look at the very high end
high_end_df = df[df['text_len'] > 5000]['text']
if len(high_end_df) > 0:
    # If there are fewer than 50 high-end texts, show all of them.
    # Otherwise, sample 50.
    sample_count = min(50, len(high_end_df))
    print(high_end_df.sample(sample_count))
else:
    print("No texts found with length greater than 5000 characters.")

Posts with fewer than 20 characters are not providing a useful signal. Posts with more than 5000 characters are also not useful because they look like blank spaces or spam.

In [ ]:
# Drop rows where text_len > 5000 or < 20
df = df[(df['text_len'] >= 20) & (df['text_len'] <= 5000)]

In [ ]:
print(f"Rows remaining: {len(df)}")
print(f"text_len range: {df['text_len'].min()} – {df['text_len'].max()}")
print(df['class'].value_counts())  # check class balance held up

In [ ]:
# Look at the new low end
df[df['text_len'] < 50]['text'].sample(20)

In [ ]:
# Look at the new high end
high_end_df = df[df['text_len'] > 4000]['text']
if len(high_end_df) > 0:
    # If there are fewer than 50 high-end texts, show all of them.
    # Otherwise, sample 50.
    sample_count = min(50, len(high_end_df))
    print(high_end_df.sample(sample_count))
else:
    print("No texts found with length greater than 5000 characters.")

Posts with fewer than 50 characters still seem to not be providing much of a useful signal, but since the shorter posts skew to the non-suicide class, they can be left in. Posts with more than 4000 characters starting to look more useful. Many of them are padded with extra spaces that will be removed during cleaning.

In [ ]:
# random sample to get a subset of 100,000 rows of data for our project, use randon_state to make sure it is the same subset
# df = df.sample(100000,random_state=42)
df = df.sample(10000,random_state=42)
df

In [ ]:
# check balance of classes after taking a smaller sample
df['class'].value_counts()

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(nltk.corpus.stopwords.words('english'))
punctuations = string.punctuation

In [ ]:
def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [ ]:
# Function to clean_text deep clean
def clean_text(text):
    text = re.sub(r'filler filler+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'fillerfiller+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    text = text.translate(str.maketrans('', '', punctuations)).lower() #lowercase
    raw_tokens = nltk.word_tokenize(text)
    tagged_tokens = nltk.pos_tag(raw_tokens)
    tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged_tokens
        if word not in stop_words and not word.isdigit()
    ]
    return ' '.join(tokens)

In [ ]:
df['text_clean_deep'] = df['text'].apply(clean_text)

In [ ]:
# Function to clean_text
def clean_for_deep_learning(text):
    text = re.sub(r'filler filler+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'fillerfiller+', '', text, flags=re.IGNORECASE)
    text = str(text)
    text = re.sub(r'\s+', ' ', text).lower() #lowercase
    return text.strip()

In [ ]:
%%time

df['text_clean_mild'] = df['text'].apply(clean_for_deep_learning)

In [ ]:
df[['text_clean_deep','text_clean_mild','text']].sample(5)

In [ ]:
# Verify that no cells in cleaned combined text column are null
df['text_clean_deep'].isnull().sum()

In [ ]:
# notice that there are one or more non-english records: remove all non-English records
def detect_english(text):
    try:
        return detect(str(text)[:200]) == 'en'
    except:
        return False
df['is_en'] = df['text'].apply(detect_english)
df_clean = df[df['is_en'] == True].copy()
df_clean = df_clean.drop(columns=['is_en'])

In [ ]:
df_clean[['text_clean_deep','text_clean_mild','text']].sample(5)
df_clean['label'] = df_clean['class'].map({'non-suicide': 0, 'suicide': 1})
df_clean.to_csv('df_clean.csv', index=False)

Sentiment Analysis **(VADER)**

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

In [ ]:
nltk.download('vader_lexicon')

In [ ]:
# create analyzer
sia = SentimentIntensityAnalyzer()

In [ ]:
# get sentiment score for each post
df_clean['sentiment_score'] = df_clean['text_clean_mild'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

print("Sample sentiment scores:")
df_clean[['text', 'class', 'sentiment_score']].sample(5)

In [ ]:
# compare sentiment by class
sentiment_summary = df_clean.groupby('class')['sentiment_score'].mean()

print("Average sentiment by class:")
print(sentiment_summary)

In [ ]:
# distribution of sentiment scores
plt.figure(figsize=(8, 5))
plt.hist(df_clean['sentiment_score'], bins=30)

plt.title("Distribution of Sentiment Scores")
plt.xlabel("Sentiment Score")
plt.ylabel("Number of Posts")

plt.show()

In [ ]:
# sentiment distribution by class
plt.figure(figsize=(8, 5))

for label in df_clean['class'].unique():
    plt.hist(
        df_clean[df_clean['class'] == label]['sentiment_score'],
        bins=30,
        alpha=0.5,
        label=label
    )

plt.title("Sentiment Score Distribution by Class")
plt.xlabel("Sentiment Score")
plt.ylabel("Number of Posts")
plt.legend()

plt.show()

#suicidal posts tend to have lower sentiment scores on average
It’s interesting to see the distribution of the sentiment score by class… Yes, there’s a big spike in very low scores for the Suicide class, and a good spike in neutral scores for the non-suicide class, but otherwise both classes are distributed similarly.

**EDA**: word cloud for suicide_data & non_suicide_data



In [ ]:
suicide_data = df[df['class'] == 'suicide']['text_clean_deep']
non_suicide_data = df[df['class'] == 'non-suicide']['text_clean_deep']

In [ ]:
text_suicide = " ".join(str(i) for i in suicide_data)
text_non_suicide = " ".join(str(i) for i in non_suicide_data)

In [ ]:
def generate_wc(text, title, color):
    wc = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap=color,
        max_words=150
    ).generate(text)

    plt.imshow(wc, interpolation='bilinear')
    plt.title(title, fontsize=14)
    plt.axis('off')

In [ ]:
# Generate word cloud
plt.figure(figsize=(30, 20))
plt.subplot(1, 2, 1)
generate_wc(text_suicide, 'Suicide word Cloud', 'Reds')
# Generate word cloud
plt.subplot(1, 2, 2)
generate_wc(text_non_suicide, 'Non-Suicide Cloud', 'Greens')

plt.show()

---
## Topic Modeling with BERTopic

Before moving into classification, we apply BERTopic to explore the latent thematic structure of the corpus. Unlike the classification models below, BERTopic is **unsupervised** — it discovers topics without using the `class` label. This lets us examine whether the textual content naturally clusters in ways that align with the suicide vs. non-suicide distinction.

BERTopic chains three steps: (1) sentence embeddings via a pretrained transformer, (2) dimensionality reduction with UMAP, and (3) density-based clustering with HDBSCAN. c-TF-IDF is then used to extract the most representative words per cluster.

**Dataset scope:** BERTopic is run on the **full filtered dataset** (`df` before the 10,000-row sample) so the topic structure reflects the complete corpus. The 10,000-row sample used for classification is drawn afterwards.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)
from bertopic import BERTopic
import re

print(f'Full filtered dataset size: {len(df):,} rows')
print(df['class'].value_counts())

In [ ]:
# BERTopic works best with light cleaning — preserves semantic context
# We reuse clean_for_deep_learning (mild clean) applied to the full df
def clean_for_bertopic(text):
    text = re.sub(r'filler filler+', '', str(text), flags=re.IGNORECASE)
    text = re.sub(r'fillerfiller+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).lower()
    return text.strip()

In [ ]:
%%time
df['text_bertopic'] = df['text'].apply(clean_for_bertopic)
print(f'Cleaned {len(df):,} documents for BERTopic')

### Fit BERTopic on the full corpus

We fit a single model on all documents (both classes combined) to discover topics organically, then inspect whether topics skew toward one class.

In [ ]:
%%time

# Instantiate BERTopic with default settings (all-MiniLM-L6-v2 embeddings)
# min_topic_size controls the minimum cluster size — lower = more granular topics
topic_model = BERTopic(min_topic_size=50, verbose=True)
topics, probs = topic_model.fit_transform(df['text_bertopic'].tolist())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Update topic representations to exclude stopwords (no retraining needed)
vectorizer = CountVectorizer(stop_words='english')
topic_model.update_topics(df['text_bertopic'].tolist(), vectorizer_model=vectorizer)

In [ ]:
# Overview of discovered topics
# Topic -1 = outlier documents that don't fit any cluster
topic_info = topic_model.get_topic_info()
print(f'Total topics discovered (excl. outliers): {len(topic_info) - 1}')
print(f'Outlier documents (Topic -1): {topic_info[topic_info.Topic == -1]["Count"].values[0]:,}')
topic_info.head(15)

In [ ]:
# Top words for each of the first 10 topics
topic_model.get_topic_info()[['Topic', 'Count', 'Name']].head(11)

In [ ]:
# Visualize word bar chart for top 10 topics
topic_model.visualize_barchart(top_n_topics=10, n_words=6)

### Topic distribution by class

We now attach the predicted topic to each document and compare topic distributions across the `suicide` and `non-suicide` classes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Attach topics back to the full df
df_bt = df.copy()
df_bt['topic'] = topics

# Exclude outliers (topic -1) for distribution analysis
df_bt_filtered = df_bt[df_bt['topic'] != -1]

# Count topic assignments per class
topic_class_counts = (
    df_bt_filtered
    .groupby(['topic', 'class'])
    .size()
    .reset_index(name='count')
)

# Top 15 topics by total document count
top_topics = (
    topic_class_counts.groupby('topic')['count'].sum()
    .nlargest(15).index.tolist()
)
plot_df = topic_class_counts[topic_class_counts['topic'].isin(top_topics)]

plt.figure(figsize=(14, 6))
sns.barplot(data=plot_df, x='topic', y='count', hue='class',
            palette={'suicide': '#e74c3c', 'non-suicide': '#2ecc71'})
plt.title('Document Count per Topic by Class (Top 15 Topics)', fontsize=14, fontweight='bold')
plt.xlabel('Topic ID')
plt.ylabel('Document Count')
plt.legend(title='Class')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Proportion of each class within each topic
topic_pivot = (
    topic_class_counts[topic_class_counts['topic'].isin(top_topics)]
    .pivot(index='topic', columns='class', values='count')
    .fillna(0)
)
topic_pivot['total'] = topic_pivot.sum(axis=1)
topic_pivot['pct_suicide'] = topic_pivot['suicide'] / topic_pivot['total']
topic_pivot = topic_pivot.sort_values('pct_suicide', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
topic_pivot['pct_suicide'].plot(kind='bar', color='#e74c3c', alpha=0.8, ax=ax)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_title('% Suicide Posts per Topic (Top 15 Topics)', fontsize=14, fontweight='bold')
ax.set_xlabel('Topic ID')
ax.set_ylabel('Proportion Suicide')
ax.set_ylim(0, 1)
sns.despine()
plt.tight_layout()
plt.show()

print(topic_pivot[['suicide', 'non-suicide', 'total', 'pct_suicide']].to_string())

### Class-specific topic models

We also fit BERTopic **separately** on each class to surface the distinct themes within suicide and non-suicide posts. This provides a richer picture than the combined model alone.

In [ ]:
%%time

suicide_docs     = df[df['class'] == 'suicide']['text_bertopic'].tolist()
non_suicide_docs = df[df['class'] == 'non-suicide']['text_bertopic'].tolist()

topic_model_suicide     = BERTopic(min_topic_size=40, verbose=False)
topic_model_non_suicide = BERTopic(min_topic_size=40, verbose=False)

topics_s,  _ = topic_model_suicide.fit_transform(suicide_docs)
topics_ns, _ = topic_model_non_suicide.fit_transform(non_suicide_docs)

print(f'Suicide topics:     {len(topic_model_suicide.get_topic_info()) - 1}')
print(f'Non-suicide topics: {len(topic_model_non_suicide.get_topic_info()) - 1}')

In [ ]:
# Update class-specific models to exclude stopwords
topic_model_suicide.update_topics(suicide_docs, vectorizer_model=CountVectorizer(stop_words='english'))
topic_model_non_suicide.update_topics(non_suicide_docs, vectorizer_model=CountVectorizer(stop_words='english'))

In [ ]:
print('=== Top Suicide Topics ===')
display(topic_model_suicide.get_topic_info().head(11))

print('\n=== Top Non-Suicide Topics ===')
display(topic_model_non_suicide.get_topic_info().head(11))

In [ ]:
# Suicide posts — top topics
fig_suicide = topic_model_suicide.visualize_barchart(top_n_topics=5, n_words=6)
fig_suicide.update_layout(title_text='Suicide Posts — Top Topics')
fig_suicide.show()

In [ ]:
# Non-suicide posts — top topics
fig_non_suicide = topic_model_non_suicide.visualize_barchart(top_n_topics=5, n_words=6)
fig_non_suicide.update_layout(title_text='Non-Suicide Posts — Top Topics')
fig_non_suicide.show()

### Reduce to top N topics

To simplify interpretation, we reduce the combined model to the top 10 topics.

In [ ]:
# Reduce combined model to top 10 topics
topic_model_reduced = topic_model.reduce_topics(df['text_bertopic'].tolist(), nr_topics=10)

print('Reduced topic info:')
topic_model_reduced.get_topic_info()

In [ ]:
topic_model_reduced.visualize_barchart(n_words=6)

### BERTopic Summary

BERTopic reveals the latent thematic structure of the corpus without using any labels. Topics dominated by suicidal ideation language (e.g., clusters around self-harm, hopelessness, or crisis) that naturally separate from non-suicide topics suggest the content itself is semantically distinct — supporting the feasibility of the classification task that follows.

The class-specific models offer an additional lens: what themes are most central *within* each class, independent of the other.

---